In [0]:
from pyspark.sql.functions import *



In [0]:
SOURCE_TABLE = "workspace.silver.historical_hourly"

TARGET_TABLE = "workspace.gold.city_weather_summary"

silver_df = spark.table(SOURCE_TABLE)

In [0]:
city_summary_df = (
    silver_df
    .groupBy("city_name")
    .agg(
        avg("temperature").alias("avg_temperature"),
        max("temperature").alias("highest_temperature"),
        min("temperature").alias("lowest_temperature"),

        avg("humidity").alias("avg_humidity"),

        avg("wind_speed").alias("avg_wind_speed"),
        max("wind_speed").alias("highest_wind_speed"),

        sum("precipitation_lwe").alias("total_precipitation"),

        avg("visibility").alias("avg_visibility"),

        avg("index_uv").alias("avg_uv_index"),

        count("*").alias("total_observations")
    )
)

In [0]:
city_summary_df = (
    city_summary_df
    .withColumn("avg_temperature", round("avg_temperature",2))
    .withColumn("avg_humidity", round("avg_humidity",2))
    .withColumn("avg_wind_speed", round("avg_wind_speed",2))
    .withColumn("avg_visibility", round("avg_visibility",2))
    .withColumn("avg_uv_index", round("avg_uv_index",2))
    .withColumn("total_precipitation", round("total_precipitation",2))
)

In [0]:
(
    city_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)